In [1]:
# import
import psycopg2
import pandas as pd

In [2]:
# connect to postgresql
conn = psycopg2.connect(
    host = "localhost",
    database = "postgres",
    user = "postgres",
    password = "Codio590",
    port=5432)

print("Connected to Postgres")

Connected to Postgres


In [3]:
# verify connection / test
cur = conn.cursor()

cur.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
""")

print(cur.fetchall())

[('gpcnme',), ('fdes',), ('nutval',), ('ctgnme',), ('nutdes',), ('wght',)]


In [11]:
# Queries to populate dashboard

# pull food + category + nutrient values
query = """
    SELECT f."Descriptor" AS food,
        c."Category description" AS category,
        n."Nutrient description" AS nutrient,
        v."Nutrient value" AS nutrient_value,
        n."Nutrient unit" AS unit
    FROM fdes f
    JOIN ctgnme c
        ON f."Food category code" = c."Food category code"
    JOIN nutval v
        ON f."Cn code" = v."Cn code"
    JOIN nutdes n
        ON v."Nutrient code" = n."Nutrient code"
    WHERE v."Nutrient value" IS NOT NULL
    """

df = pd.read_sql(query, conn)
df.head()

/tmp/ipykernel_6386/2723914593.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,food,category,nutrient,nutrient_value,unit
0,"Butter, salted",Dairy and Egg Products,Protein,0.85,g
1,"Butter, salted",Dairy and Egg Products,Total Fat,81.11,g
2,"Butter, salted",Dairy and Egg Products,Carbohydrate,0.06,g
3,"Butter, salted",Dairy and Egg Products,Ash,2.11,g
4,"Butter, salted",Dairy and Egg Products,Food Energy,717.00,kcal


In [13]:
# install dash
%pip install dash

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from dash import Dash, dcc, html, Input, Output, dash_table

In [33]:
# create dashboard

app = Dash(__name__)

# grab the query without any duplicates and without blanks
nutrients = sorted(df["nutrient"].dropna().unique())

app.layout = html.Div([
    # add header
    html.H1("Nutritional Dashboard"),
    html.P("Search foods by nutrient and limit results by a second nutrient."),

    # add nutrient drop down
    html.Label("Select Nutrient",
        style = {"fontWeight": "bold"}),

    dcc.Dropdown(
        id = "nutrient_dropdown",
        options = [{"label": n, "value": n} for n in nutrients],
        value = nutrients[0],
        clearable = False ),

    # add limiting nutrient drop down
    html.Label("Nutrient to Limit",
        style = {"fontWeight": "bold"}),

    dcc.Dropdown(
        id = "limit_nutrient_dropdown",
        options = [{"label": n, "value": n} for n in nutrients], 
        value = nutrients[0],
        clearable = False ),

    html.Label("Maximum Amount",
        style = {"fontWeight": "bold"}),

    dcc.Input(
        id = "limit_value",
        type = "number",
        value = 15,
        style = {"width": "120px"}),

    html.Br(),

    # data table for the dashboard
    dash_table.DataTable(
        id = "results",
        columns = [
            {"name": "Food", "id": "food"},
            {"name": "Category", "id": "category"},
            {"name": "Nutrient", "id": "nutrient"},
            {"name": "Value", "id": "nutrient_value"},
            {"name": "Unit", "id": "unit"},
            {"name": "Limit Nutrient Value", "id": "limit_value"}],
        page_size = 10,
        style_table = {"overflowX": "auto"},
        style_cell = {"textAlign": "left", "padding": "6px"},
        style_header = {"fontWeight": "bold"},
        sort_action = "native" )])

@app.callback(
    Output("results", "data"),
    Input("nutrient_dropdown", "value"),
    Input("limit_nutrient_dropdown", "value"),
    Input("limit_value", "value"))

# update results table for selected nutrients and limitations
def update_table(for_nutrient, limit_nutrient, limit_value):

    main = df[df["nutrient"] == for_nutrient].copy()

    limit = df[df["nutrient"] == limit_nutrient][
        ["food", "category", "nutrient_value"]].copy()

    limit = limit.rename(columns = {"nutrient_value": "limit_value"})

    merged = main.merge(limit, on = ["food", "category"], how = "left")

    # limit results based on input
    if limit_value is not None:
        merged = merged[merged["limit_value"] <= limit_value]

    merged = merged.sort_values("nutrient_value", ascending = False)

    return merged.head(25).to_dict("records")

app.run(jupyter_mode = "inline", port = 8051)  

    